To rerun the training process from scratch, delete the folders model, feature_store, and delete the files train_embeddings.csv, interaction_heatmap.png, and transformer.pth

Accordingly move datasets into the data folder depending on what data you want to test/train on, and name them test.csv and train.csv respectively

In [3]:
%%time
!python generate_features.py

Beginning feature generation based on train/test split...
Total unique molecules to process: 732 solutes, 128 solvents.
Generating ALL Solute Raw Features...
Computing features for 732 molecules...
Featurizing: 100%|███████████████████████████| 732/732 [00:06<00:00, 107.87it/s]
Generating ALL Solvent Raw Features...
Computing features for 128 molecules...
Featurizing: 100%|███████████████████████████| 128/128 [00:00<00:00, 364.44it/s]

Determining Training Distribution to FIT the Council Extractor...
Council Extractor fitted on distribution defined by top 19 solvents.
Transforming ALL molecules into Council Features...
Generated all required feature stores for the entire dataset, scaled using only the training distribution.
CPU times: user 60.5 ms, sys: 25 ms, total: 85.4 ms
Wall time: 9.32 s


In [4]:
%%time
!python train_transformer.py

Starting 5-Fold OOF Training on mps...
Training on Fold 1: 100%|███████████████████████| 20/20 [01:00<00:00,  3.00s/it]
  Fold 1 complete.
Training on Fold 2: 100%|███████████████████████| 20/20 [01:00<00:00,  3.01s/it]
  Fold 2 complete.
Training on Fold 3: 100%|███████████████████████| 20/20 [00:59<00:00,  2.99s/it]
  Fold 3 complete.
Training on Fold 4: 100%|███████████████████████| 20/20 [01:00<00:00,  3.05s/it]
  Fold 4 complete.
Training on Fold 5: 100%|███████████████████████| 20/20 [00:57<00:00,  2.89s/it]
  Fold 5 complete.
Training Final Production Model...
Final Pass: 100%|███████████████████████████████| 20/20 [01:14<00:00,  3.70s/it]
Success: train_embeddings.csv, transformer.pth, and interaction_heatmap.png saved.
CPU times: user 2.27 s, sys: 852 ms, total: 3.12 s
Wall time: 6min 41s


In [5]:
%%time
!python train.py

Training model
Initial features: 380
Pruned features:  289
Physical consistency maintained

TRAINING WITH 5 SEEDS

--- Seed 42 ---
0:	learn: 1.1583031	test: 1.1605958	best: 1.1605958 (0)	total: 114ms	remaining: 5m 43s
500:	learn: 0.4507225	test: 0.4662884	best: 0.4662884 (500)	total: 25.1s	remaining: 2m 5s
1000:	learn: 0.3219639	test: 0.3376208	best: 0.3376208 (1000)	total: 52.1s	remaining: 1m 44s
1500:	learn: 0.2648303	test: 0.2806727	best: 0.2806727 (1500)	total: 1m 17s	remaining: 1m 17s
2000:	learn: 0.2325318	test: 0.2484040	best: 0.2484040 (2000)	total: 1m 44s	remaining: 52s
2500:	learn: 0.2129926	test: 0.2283138	best: 0.2283138 (2500)	total: 2m 10s	remaining: 26.1s
2999:	learn: 0.2013678	test: 0.2177951	best: 0.2177890 (2998)	total: 2m 39s	remaining: 0us

bestTest = 0.2177889675
bestIteration = 2998

Shrink model to first 2999 iterations.
  RMSE: 0.6785, R²: 0.6574

--- Seed 101 ---
0:	learn: 1.1583057	test: 1.1651718	best: 1.1651718 (0)	total: 54.8ms	remaining: 2m 44s
500:	learn:

In [6]:
%%time
!python eval_test.py

Evaluation testing
Loading model artifacts...
Generating learned interactions for 9137 molecules...
Applying Thermodynamic Engineering...
Performing final predictions...

Test Results
RMSE: 0.6745
R2: 0.6615
Inference timing results:
Total Time: 0.4422 s
Throughput: 20664 pairs/sec
CPU times: user 27.6 ms, sys: 15.1 ms, total: 42.7 ms
Wall time: 4.56 s


In [7]:
!python ablations.py

GLASS ONION ABLATION STUDY

Loading data and features...

Total features: 380
x_A: 0-176 (176 features)
x_B: 176-352 (176 features)
I: 352-376 (24 features)
f(T): 376-380 (4 features: Tm, T_red, T, T_inv)

BASELINE VARIANCE ESTIMATION (5 seeds)
  Seed 42... R²=0.6574, RMSE=0.6785
  Seed 101... R²=0.6577, RMSE=0.6782
  Seed 123... R²=0.6615, RMSE=0.6745
  Seed 456... R²=0.6576, RMSE=0.6783
  Seed 789... ^C
Traceback (most recent call last):
  File "/Users/ashish/Documents/glass-onion/regime-ii/ablations.py", line 524, in <module>
    run_ablation_study()
  File "/Users/ashish/Documents/glass-onion/regime-ii/ablations.py", line 400, in run_ablation_study
    baseline_r2_mean, baseline_r2_std, baseline_rmse_mean, baseline_rmse_std = run_baseline_multiseed(
  File "/Users/ashish/Documents/glass-onion/regime-ii/ablations.py", line 265, in run_baseline_multiseed
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
  File "/Users/ashish/miniforge3/envs/molmerger/lib/python3.10/site-packages/cat

# Hyperparameter sweep

BigSol 1.0

In [ ]:
!python hyperparam_sweep.py

HYPERPARAMETER OPTIMIZATION - CATBOOST
Trials: 30, Validation Split: 15%

[1] Loading train CSV...
  Loaded 38540 rows
  Loading OOF embeddings...
  Loaded OOF: 38540 rows
  Generating hyper features (this may take a while)...
  DONE! Feature dim: (38540, 380)
  Train samples: 38540
  Feature dim: 380

[2] Applying feature selection...
  Loading selector...
  Transforming features...
  Selected features: 289

[3] Splitting data...
  Train: 32759, Val: 5781

[4] Running Optuna optimization...
[I 2026-01-19 13:42:32,402] A new study created in memory with name: no-name-0f5b07a1-ff82-4e8b-945c-cbe598f72d49
  0%|                                                    | 0/30 [00:00<?, ?it/s]
>>> Trial 0: depth=9, iter=3500, lr=0.0896
0:	learn: 1.1189484	test: 1.1235686	best: 1.1235686 (0)	total: 178ms	remaining: 10m 23s
100:	learn: 0.3910508	test: 0.4109236	best: 0.4109236 (100)	total: 10.2s	remaining: 5m 44s
200:	learn: 0.2853789	test: 0.3087882	best: 0.3087882 (200)	total: 20.1s	remaining: 5m